<a href="https://colab.research.google.com/github/RytisBalt/Ma-ininis-mokymasis/blob/TreciasKontrolinis/TMMA2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import toeplitz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rho = 0.99
n_features = 150
n_samples = 600
pcg_signif = 0.30
n_signif = int(n_features * pcg_signif)

beta = [2] * (n_signif) + [0] * (n_features - n_signif)

cov_matrix = toeplitz([rho**i for i in range(n_features)])

# Set seed for reproducibility
np.random.seed(42)
# Generate X and y
X = np.random.multivariate_normal(np.zeros(n_features), cov_matrix, size=n_samples)
noise = np.random.normal(0, 10, n_samples)
y = X.dot(beta) + noise

# Custom rescaling of certain X columns
X[:, 0] = X[:, 0] * 100
X[:, 1:5] = X[:, 1:5] * 400
X[:, 5:10] = X[:, 5:10] * 1000
X[:, 10:20] = X[:, 10:20] * 1500


X = pd.DataFrame(X)
y = pd.Series(y)

# Train/test split:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)


In [ ]:
# modeliai


from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

krr = GridSearchCV(
    KernelRidge(kernel="linear"),
    param_grid={"alpha": [1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3]},
)
svr = GridSearchCV(
    estimator=LinearSVR(),
    param_grid={"C": [1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3], "epsilon": [1e-2,1e-1,5e-1,1e0]},
)
las = LassoCV()
#Dokumentacijoje parasyta, kad alphas yra reguliarizacijos parametras ir jei jį
#paliksi kaip none, tai ims automatiškai, manau tai ir turimą omenyje.


In [ ]:
krr.fit(X_train_scaled, y_train)
svr.fit(X_train_scaled, y_train)
las.fit(X_train_scaled, y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

LassoCV()

In [ ]:
#2 uzduotis
print(f"Geriausias KRR: {krr.best_params_}")
print(f"Geriausias SVR: {svr.best_params_}")
print(f"Geriausias Lasso alpha: {las.alpha_}")
#Geriausi parinkti parametrai

Geriausias KRR: {'alpha': 100.0}
Geriausias SVR: {'C': 0.1, 'epsilon': 0.01}
Geriausias Lasso alpha: 0.2041662270639378


In [ ]:
#3 uzduotis
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import r2_score as r2
lst = []

for clf, name in zip([krr, svr, las], ['krr', 'svr', 'las']):
    y_pred = clf.predict(X_test_scaled)
    lst.append([mse(y_pred, y_test), r2(y_pred, y_test)])

df = pd.DataFrame(lst, columns=['MSE', "R^2"], index=['krr', 'svr', 'las'])
print(df)

            MSE       R^2
krr   99.938165  0.983962
svr  101.582053  0.983480
las   96.649312  0.984451


In [ ]:
#4 uzduotis
nenuliai = np.sum(las.coef_ != 0)
print("Koeficientai, kurie reikšmės nelygu nuliui:", nenuliai)

mse_path_mean = np.mean(las.mse_path_, axis=1)
mse_path_std = np.std(las.mse_path_, axis=1)
one_se_threshold = mse_path_mean[np.argmin(mse_path_mean)] + mse_path_std[np.argmin(mse_path_mean)]

alpha_1se = las.alphas_[np.where(mse_path_mean <= one_se_threshold)[0][-1]]

las_1se = LassoCV(alphas=[alpha_1se]).fit(X_train_scaled, y_train)
non_zero_coeffs_1se = np.sum(las_1se.coef_ != 0)
print("Koeficientai, kurių reikšmės nelygui nuliui po 1-SE taisyklės:", non_zero_coeffs_1se)

#Gavome rezulatus 37 originalus ir 45 po 1-SE. Mano manymu ir surasta informacija, kad
#originalus modelis yra retesnis, nes jis turi mažiau nenulinių reikšmių negu modelis, kuriam
#buvo pritaikyta 1-SE taisyklė. Kurį modelį reikėtų rinktis, čia priklauso nuo tyrėjo, jeigu
#neturi tokių didelių resursų palaikančio kompiuterio, gal geriau būtų imti originalų, bet jei
#nori tikslesnio spėjimo rezultato, imčiau las_1se

Koeficientai, kurie reikšmės nelygu nuliui: 37
Koeficientai, kurių reikšmės nelygui nuliui po 1-SE taisyklės: 45


In [ ]:
# Randame nenulinius koeficientus:
coef = las.coef_
selected_features = np.where(coef != 0)[0]
# Atrenkame požymių poaibį kaip reikšmingą:
X_train_selected = X_train_scaled[:, selected_features]
X_test_selected = X_test_scaled[:, selected_features]

In [ ]:
# Ieškome antrą kartą parametrų:
relaxed_las_cv = LassoCV().fit(X_train_selected, y_train)

print(f"Best alpha using built-in LassoCV: {relaxed_las_cv.alpha_}")

# Fit: (Nebūtinas, bet galima pasikartoti)
relaxed_lasso = Lasso(alpha=relaxed_las_cv.alpha_)
relaxed_lasso.fit(X_train_selected, y_train)

Best alpha using built-in LassoCV: 0.08242225822796592


Lasso(alpha=0.08242225822796592)

In [ ]:
# Prognozės
y_pred_relaxed_lasso = relaxed_lasso.predict(X_test_selected)
print(f"Mean squared error: {mse(y_test, y_pred_relaxed_lasso)}")

# Galime pasitikrinti, kad prognozės identiškos:
y_pred_2 = relaxed_las_cv.predict(X_test_selected)

print(f"Mean squared error: {mse(y_test, y_pred_2)}")
print("R^2 score", r2(y_test, y_pred_2))

Mean squared error: 100.85822738220068
Mean squared error: 100.85822738220068
R^2 score 0.9839155910331661


In [ ]:
from sklearn.pipeline import Pipeline

param_grid = {
    "gamma": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 2, 3],
    "alpha": [0.1, 0.2, 0.5, 1, 2, 3]
}

initial_weights = 1 / (np.abs(las.coef_) + 1e-4)

def adaptive_lasso_fit(alpha, gamma):
    weighted_X_train = X_train_scaled / (initial_weights ** gamma)
    weighted_X_test = X_test_scaled / (initial_weights ** gamma)
    lasso_model = Lasso(alpha=alpha, random_state=42).fit(weighted_X_train, y_train)
    return lasso_model, weighted_X_test

best_params = None
best_score = float('inf')

for alpha in param_grid['alpha']:
    for gamma in param_grid['gamma']:
        weighted_X_train = X_train_scaled / (initial_weights ** gamma)
        weighted_X_test = X_test_scaled / (initial_weights ** gamma)


        lasso_model = Lasso(alpha=alpha).fit(weighted_X_train, y_train)
        score = -lasso_model.score(weighted_X_train, y_train)

        if score < best_score:
            best_score = score
            best_params = {'alpha': alpha, 'gamma': gamma}

best_alpha = best_params['alpha']
best_gamma = best_params['gamma']

final_weighted_X_train = X_train_scaled / (initial_weights ** best_gamma)
final_weighted_X_test = X_test_scaled / (initial_weights ** best_gamma)

final_model = Lasso(alpha=best_alpha).fit(final_weighted_X_train, y_train)


adaptive_lasso_predictions = final_model.predict(final_weighted_X_test)
test_mse = mse(y_test, adaptive_lasso_predictions)
test_r2 = r2(y_test, adaptive_lasso_predictions)


print("Best Parameters (Alpha, Gamma):", best_params)
print("Best Score (Negative R^2):", best_score)
print("Adaptive Lasso Predictions (First 5):", adaptive_lasso_predictions[:5])
print("Test MSE", test_mse)
print("Test R^2:", test_r2)


Best Parameters (Alpha, Gamma): {'alpha': 0.1, 'gamma': 0.1}
Best Score (Negative R^2): -0.9877874105979076
Adaptive Lasso Predictions (First 5): [  1.87909908   2.97678013  32.31862109  13.83432259 -41.53073843]
Test MSE 100.07936501130146
Test R^2: 0.9840398004430236


Palyginsime visus tris modelius tarpusavyje.
 **Lasso** modelis
*   R^2 =  0,984451
*   MSE = 96,649312

**Relaxed_Lasso**

*   MSE = 100,85822738220068
*   R^2 = 0,9839155910331661,

**Adaptive_Lasso**

*   MSE = 100,07936501130146
*   R^2 = 0,9840398004430236

Pagal šituos kriterijus, aš naudočiausi originalių Lasso modeliu.